<h1 align="center"> Lingüística Computacional - MIARFID</h1>
<h1 align="center"> Trabajo final - Modelos clásicos</h1>

**Estudiantes:** Carlos Broch Cubedo · Ferran Artero Merino

# 0. Configuración

In [1]:
!pip -q install iterative-stratification

In [2]:
import os, re, ast, json, time, zipfile, itertools, warnings
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.pipeline import FeatureUnion
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.multiclass import OneVsRestClassifier
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.model_selection import ParameterGrid
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, hamming_loss, f1_score
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit, MultilabelStratifiedKFold
from google.colab import drive

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 120)
SEED = 42
N_JOBS = -1

drive.mount("/content/drive")
DATA_DIR = "/content/drive/MyDrive/LC_TrabajoFinal"
OUT_DIR = os.path.join(DATA_DIR, "outputs", "clasicos")
os.makedirs(OUT_DIR, exist_ok=True)
TRAIN_FILE = os.path.join(DATA_DIR, "dataset_train.csv")
TEST_FILE = os.path.join(DATA_DIR, "dataset_test.csv")

# True: se ejecuta el experimento y se guarda su CSV en OUT_DIR. False: se carga el CSV guardado.
RUN = {"grid_palabras": False, "dedup_jerarquico": False, "jerarquico_union": False,
       "grid_caracteres": False, "wordpiece": False, "holdout": False, "envio": False}

Mounted at /content/drive


In [3]:
MIN_DF, MAX_DF, MAX_FEATURES = 2, 0.9, 300_000
CHAR_RANGE, CHAR_MIN_DF, CHAR_MAX_FEATURES = (2, 5), 3, 300_000
WORDPIECE_VOCAB = 8000
SMART_MINORITY_MAX = 1000     # clase minoritaria: menos de 1000 artículos únicos
HIER_RULE_MIN_P = 0.5         # la secundaria s asigna la raíz r si P(r | s) >= 0.5
MAX_LABELS = 4
HOLDOUT_SIZE, N_FOLDS = 0.15, 3

NGRAMS = [(1, 1), (1, 2), (1, 3)]
GRID = {"NB":  {"variant": ["multinomial", "complement"], "alpha": [0.01, 0.03, 0.1, 0.3, 1.0]},
        "LR":  {"C": [2, 8, 32]},
        "SVM": {"C": [0.05, 0.1, 0.3, 1.0]}}

BASE = {"dedup": "smart", "hier": False, "root_mode": "fallback", "char": False, "tokenizer": "regex"}

# 1. Datos
Las filas se agrupan por título y resumen normalizados, uniendo las etiquetas de las copias. El texto de entrada es `título. resumen`.

In [4]:
CLASSES = ['cs.AI', 'cs.CL', 'cs.CR', 'cs.CV', 'cs.CY', 'cs.DC', 'cs.GR', 'cs.HC', 'cs.IR', 'cs.IT', 'cs.LG',
           'cs.MA', 'cs.MM', 'cs.NE', 'cs.RO', 'cs.SI', 'cs.SY', 'eess.IV', 'eess.SP', 'eess.SY', 'math.IT',
           'math.OC', 'math.ST', 'physics.chem-ph', 'stat.AP', 'stat.ME', 'stat.ML', 'stat.TH']
L = len(CLASSES)

def norm(s):
    s = str(s).lower()
    s = re.sub(r"\$[^$]*\$", " ", s)
    s = re.sub(r"[^a-z0-9]+", " ", s)
    return re.sub(r"\s+", " ", s).strip()

def clean(s):
    return re.sub(r"\s+", " ", str(s)).strip()

raw = pd.read_csv(TRAIN_FILE)
raw["labels"] = raw["terms"].map(ast.literal_eval)
raw["title_n"], raw["summary_n"] = raw["titles"].map(norm), raw["summaries"].map(norm)

data = (raw.groupby(["title_n", "summary_n"], sort=False)
           .agg(titles=("titles", "first"), summaries=("summaries", "first"),
                labels=("labels", lambda ls: sorted({l for x in ls for l in x})),
                n_copies=("titles", "size"))
           .reset_index(drop=True))
data["text"] = data["titles"].map(clean) + ". " + data["summaries"].map(clean)
TEXTS = data["text"].to_numpy()

mlb = MultiLabelBinarizer(classes=CLASSES)
Y = mlb.fit_transform(data["labels"])
N = len(data)
N_COPIES = data["n_copies"].to_numpy()
print(f"Filas: {len(raw)} | artículos únicos: {N}")

Filas: 43555 | artículos únicos: 32577


In [5]:
ROOTS = ["cs.CV", "cs.LG", "stat.ML"]
ROOT_IDX = np.array([CLASSES.index(r) for r in ROOTS])
SEC = [c for c in CLASSES if c not in ROOTS]
SEC_IDX = np.array([CLASSES.index(c) for c in SEC])
ALIAS_PAIRS = [(a, b) for i, a in enumerate(CLASSES) for b in CLASSES[i + 1:]
               if (Y[:, CLASSES.index(a)] == Y[:, CLASSES.index(b)]).all()]

freq = pd.Series(Y.sum(0), index=CLASSES)
MINORITY_IDX = np.array([CLASSES.index(c) for c in freq.index[freq < SMART_MINORITY_MAX]])
print("Alias:", ALIAS_PAIRS)
print(f"Artículos sin secundaria: {100 * (Y[:, SEC_IDX].sum(1) == 0).mean():.1f} %")

Alias: [('cs.IT', 'math.IT'), ('cs.SY', 'eess.SY'), ('math.ST', 'stat.TH')]
Artículos sin secundaria: 70.7 %


### Deduplicado
Copias de cada artículo que entran en el entrenamiento: todas (`none`), una (`full`) o todas solo si el artículo tiene alguna clase minoritaria (`smart`).

In [6]:
def copies_to_keep(mode):
    if mode == "none":
        return N_COPIES.copy()
    if mode == "full":
        return np.ones(N, dtype=int)
    return np.where(Y[:, MINORITY_IDX].any(1), N_COPIES, 1)

dedup_tab = pd.DataFrame({m: Y.T @ copies_to_keep(m) for m in ("none", "full", "smart")}, index=CLASSES)
dedup_tab.loc["TOTAL FILAS"] = [copies_to_keep(m).sum() for m in ("none", "full", "smart")]
display(dedup_tab.sort_values("full", ascending=False).T)

,TOTAL FILAS,cs.CV,cs.LG,stat.ML,cs.AI,eess.IV,cs.RO,cs.CL,cs.NE,cs.CR,...,cs.SY,eess.SY,stat.ME,stat.AP,stat.TH,math.ST,math.IT,cs.IT,cs.HC,physics.chem-ph
none,43555,27078,22604,12064,6041,1823,1340,1053,639,328,...,81,81,55,59,49,49,49,49,48,40
full,32577,19787,17166,9567,4498,1261,1055,704,509,274,...,69,69,47,46,43,43,42,42,40,20
smart,33721,20167,18078,9976,4657,1261,1055,1053,639,328,...,81,81,55,59,49,49,49,49,48,40


# 2. Partición
Holdout del 15 % de los artículos únicos y 3 particiones estratificadas sobre el 85 % restante (dev).

In [7]:
dev_idx, ho_idx = next(MultilabelStratifiedShuffleSplit(n_splits=1, test_size=HOLDOUT_SIZE, random_state=SEED)
                       .split(np.zeros(N), Y))
Y_dev, Y_ho = Y[dev_idx], Y[ho_idx]

FOLDS = list(MultilabelStratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
             .split(np.zeros(len(dev_idx)), Y_dev))
FOLD_ID = np.empty(len(dev_idx), dtype=int)
for k, (_, va) in enumerate(FOLDS):
    FOLD_ID[va] = k
print(f"dev: {len(dev_idx)} | holdout: {len(ho_idx)} | particiones: {[len(va) for _, va in FOLDS]}")

dev: 27674 | holdout: 4903 | particiones: [9236, 9247, 9191]


# 3. Funciones
Vectorización, clasificadores, umbral por clase con *cross-fitting*, posproceso (al menos una raíz, máximo 4 etiquetas, alias juntos) y enfoque jerárquico.

In [8]:
def compute_metrics(labels, preds):
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="macro", zero_division=0)
    return {"accuracy": accuracy_score(labels, preds), "f1": f1, "precision": precision,
            "recall": recall, "hamming_loss": hamming_loss(labels, preds)}

_TOKENIZERS = {}
def get_tokenizer(name):
    if name == "regex":
        return None
    if name not in _TOKENIZERS:
        from tokenizers import Tokenizer, models, trainers, pre_tokenizers, normalizers
        tk = Tokenizer(models.WordPiece(unk_token="[UNK]"))
        tk.normalizer = normalizers.BertNormalizer(lowercase=True)
        tk.pre_tokenizer = pre_tokenizers.BertPreTokenizer()
        tk.train_from_iterator(TEXTS.tolist(), trainers.WordPieceTrainer(vocab_size=WORDPIECE_VOCAB, special_tokens=["[UNK]"]))
        _TOKENIZERS[name] = lambda s: tk.encode(s).tokens
    return _TOKENIZERS[name]

def make_vectorizer(cfg):
    tok = get_tokenizer(cfg["tokenizer"])
    word_kw = dict(tokenizer=tok, token_pattern=None, lowercase=False) if tok else dict(lowercase=True)
    word = TfidfVectorizer(ngram_range=tuple(cfg["ngram"]), min_df=MIN_DF, max_df=MAX_DF, max_features=MAX_FEATURES,
                           sublinear_tf=True, strip_accents="unicode", **word_kw)
    if not cfg["char"]:
        return word
    char = TfidfVectorizer(analyzer="char_wb", ngram_range=CHAR_RANGE, min_df=CHAR_MIN_DF, max_df=MAX_DF,
                           max_features=CHAR_MAX_FEATURES, sublinear_tf=True, lowercase=True, strip_accents="unicode")
    return FeatureUnion([("word", word), ("char", char)])

def make_clf(model, params, multiclass=False):
    if model == "NB":
        base = (MultinomialNB if params["variant"] == "multinomial" else ComplementNB)(alpha=params["alpha"])
    elif model == "LR":
        base = (LogisticRegression(C=params["C"], max_iter=2000) if multiclass else
                LogisticRegression(C=params["C"], solver="liblinear", class_weight="balanced", max_iter=1000))
    else:
        base = LinearSVC(C=params["C"], class_weight=None if multiclass else "balanced")
    if multiclass:
        return base
    return OneVsRestClassifier(base, n_jobs=1 if model == "NB" else N_JOBS)

def class_scores(clf, X):
    # SVM y LR: decision_function; NB: log P(1|x) - log P(0|x). Umbral natural = 0
    if hasattr(clf, "decision_function"):
        return clf.decision_function(X).astype(np.float64)
    lps = [est.predict_log_proba(X) for est in clf.estimators_]
    return np.column_stack([lp[:, 1] - lp[:, 0] for lp in lps])

def tie_alias(S, classes):
    S = S.copy()
    for a, b in ALIAS_PAIRS:
        if a in classes and b in classes:
            i, j = classes.index(a), classes.index(b)
            S[:, i] = S[:, j] = (S[:, i] + S[:, j]) / 2
    return S

In [9]:
def tune_thresholds(Yt, S, n_grid=200):
    thr = np.zeros(S.shape[1])
    for j in range(S.shape[1]):
        y, s = Yt[:, j].astype(bool), S[:, j]
        cand = np.unique(np.r_[np.quantile(s, np.linspace(0, 1, n_grid)), 0.0])
        pred = s[None, :] >= cand[:, None]
        tp = (pred & y).sum(1)
        fp = (pred & ~y).sum(1)
        fn = y.sum() - tp
        thr[j] = cand[(2 * tp / np.maximum(2 * tp + fp + fn, 1)).argmax()]
    return thr

def crossfit_thresholds(Yt, S, fold_id):
    T = np.zeros_like(S)
    for k in np.unique(fold_id):
        m = fold_id == k
        T[m] = tune_thresholds(Yt[~m], S[~m])
    return T

def cap_labels(P, M, k):
    for i in np.where(P.sum(1) > k)[0]:
        on = np.where(P[i])[0]
        P[i, on[np.argsort(-M[i, on])[k:]]] = 0
    return P

def force_alias(P, classes):
    for a, b in ALIAS_PAIRS:
        if a in classes and b in classes:
            i, j = classes.index(a), classes.index(b)
            P[:, i] = P[:, j] = np.maximum(P[:, i], P[:, j])
    return P

def decode_flat(S, T):
    M = S - T
    P = (M >= 0).astype(int)
    no_root = P[:, ROOT_IDX].sum(1) == 0
    best = ROOT_IDX[M[:, ROOT_IDX].argmax(1)]
    P[no_root, best[no_root]] = 1
    return force_alias(cap_labels(P, M, MAX_LABELS), CLASSES)

In [10]:
# Enfoque jerárquico: clasificadores para las 25 secundarias, reglas raíz <- secundaria
# y un modelo multiclase sobre las combinaciones de raíces
def root_codes(Yb):
    R = Yb[:, ROOT_IDX]
    return R[:, 0] * 4 + R[:, 1] * 2 + R[:, 2]

def codes_to_roots(c):
    c = np.asarray(c)
    return np.column_stack([(c >> 2) & 1, (c >> 1) & 1, c & 1]).astype(int)

def rule_matrix(Yb):
    Ys, Yr = Yb[:, SEC_IDX], Yb[:, ROOT_IDX]
    Pc = (Ys.T @ Yr) / np.maximum(Ys.sum(0), 1)[:, None]
    R = (Pc >= HIER_RULE_MIN_P).astype(int)
    R[np.arange(len(SEC)), Pc.argmax(1)] = 1
    return R

def decode_hier(S_sec, T_sec, root_model, R, root_mode):
    M = S_sec - T_sec
    P_sec = force_alias(cap_labels((M >= 0).astype(int), M, MAX_LABELS - 1), SEC)
    rule_roots = ((P_sec @ R) > 0).astype(int)
    if root_mode == "fallback":
        roots = np.where(rule_roots.any(1, keepdims=True), rule_roots, root_model)
    else:
        roots = np.maximum(rule_roots, root_model)
    P = np.zeros((len(S_sec), L), dtype=int)
    P[:, SEC_IDX] = P_sec
    P[:, ROOT_IDX] = roots
    return P

def decode(res, T, cfg):
    if cfg["hier"]:
        return decode_hier(res["S"], T, res["root"], res["R"], cfg["root_mode"])
    return decode_flat(res["S"], T)

In [11]:
def fit_predict(cfg, X_tr, Y_tr, X_ev):
    X_tr, X_ev = X_tr.tocsr(), X_ev.tocsr()
    X_tr.sort_indices(); X_ev.sort_indices()
    if not cfg["hier"]:
        clf = make_clf(cfg["model"], cfg["params"]).fit(X_tr, Y_tr)
        return {"S": tie_alias(class_scores(clf, X_ev), CLASSES)}
    clf = make_clf(cfg["model"], cfg["params"]).fit(X_tr, Y_tr[:, SEC_IDX])
    root_clf = make_clf(cfg["model"], cfg["params"], multiclass=True).fit(X_tr, root_codes(Y_tr))
    return {"S": tie_alias(class_scores(clf, X_ev), SEC),
            "root": codes_to_roots(root_clf.predict(X_ev)), "R": rule_matrix(Y_tr)}

def vectorize_folds(cfg):
    texts = TEXTS[dev_idx]
    feats = []
    for tr, va in FOLDS:
        vec = make_vectorizer(cfg)
        feats.append((vec.fit_transform(texts[tr]), vec.transform(texts[va])))
    return feats

def run_cv(cfg, feats):
    keep = copies_to_keep(cfg["dedup"])[dev_idx]
    oof = {"S": np.zeros((len(dev_idx), len(SEC) if cfg["hier"] else L)),
           "root": np.zeros((len(dev_idx), 3), dtype=int), "R": {}}
    for k, (tr, va) in enumerate(FOLDS):
        rep = np.repeat(np.arange(len(tr)), keep[tr])
        res = fit_predict(cfg, feats[k][0][rep], Y_dev[tr][rep], feats[k][1])
        oof["S"][va] = res["S"]
        if cfg["hier"]:
            oof["root"][va], oof["R"][k] = res["root"], res["R"]
    return oof

def decode_oof(oof, T, cfg):
    if not cfg["hier"]:
        return decode_flat(oof["S"], T)
    P = np.zeros((len(dev_idx), L), dtype=int)
    for k in range(N_FOLDS):
        m = FOLD_ID == k
        P[m] = decode_hier(oof["S"][m], T[m] if np.ndim(T) == 2 else T, oof["root"][m], oof["R"][k], cfg["root_mode"])
    return P

def evaluate_oof(oof, cfg):
    Yt = Y_dev[:, SEC_IDX] if cfg["hier"] else Y_dev
    P = decode_oof(oof, crossfit_thresholds(Yt, oof["S"], FOLD_ID), cfg)
    out = compute_metrics(Y_dev, P)
    out["f1_std"] = float(np.std([f1_score(Y_dev[FOLD_ID == k], P[FOLD_ID == k], average="macro", zero_division=0)
                                  for k in range(N_FOLDS)]))
    out["f1_umbral0"] = f1_score(Y_dev, decode_oof(oof, 0.0, cfg), average="macro", zero_division=0)
    return out

In [12]:
def params_str(p):
    return ", ".join(f"{k}={v}" for k, v in p.items())

def parse_params(s):
    out = {}
    for kv in s.split(", "):
        k, v = kv.split("=")
        out[k] = v if k == "variant" else float(v)
    return out

def best_configs(df, base):
    rows = df.loc[df.groupby("modelo")["f1"].idxmax()]
    return {r.modelo: {**base, "model": r.modelo, "params": parse_params(r.params), "ngram": ast.literal_eval(r.ngram)}
            for r in rows.itertuples()}

def run_grid(base):
    rows = []
    for ngram in NGRAMS:
        feats = vectorize_folds({**base, "ngram": ngram})
        for model, grid in GRID.items():
            for params in ParameterGrid(grid):
                t0 = time.time()
                cfg = {**base, "model": model, "params": params, "ngram": ngram}
                m = evaluate_oof(run_cv(cfg, feats), cfg)
                rows.append({"modelo": model, "ngram": str(ngram), "params": params_str(params), **m,
                             "tiempo (s)": round(time.time() - t0, 1)})
                print(f"{model:3s} {ngram} {params_str(params):32s} F1={m['f1']:.4f} ±{m['f1_std']:.4f}")
    return pd.DataFrame(rows)

def experiment(name, fn):
    path = os.path.join(OUT_DIR, f"{name}.csv")
    if RUN[name]:
        df = fn()
        df.to_csv(path, index=False)
        return df
    return pd.read_csv(path) if os.path.exists(path) else None

METRICS = ["f1", "f1_std", "accuracy", "precision", "recall", "hamming_loss"]

# 4. Búsqueda de hiperparámetros
NB, regresión logística y SVM con n-gramas de palabras (1,1), (1,2) y (1,3): 51 configuraciones evaluadas con la F1 macro *out-of-fold*.

In [13]:
grid_pal = experiment("grid_palabras", lambda: run_grid(BASE))
tabla_31 = (grid_pal.loc[grid_pal.groupby(["modelo", "ngram"])["f1"].idxmax(),
                         ["modelo", "params", "ngram", "f1_umbral0", "f1", "f1_std"]]
            .sort_values(["modelo", "ngram"]).round(4))
display(tabla_31)
BEST_WORD = best_configs(grid_pal, BASE)

,modelo,params,ngram,f1_umbral0,f1,f1_std
10,LR,C=2,"(1, 1)",0.2992,0.3184,0.0071
28,LR,C=8,"(1, 2)",0.2402,0.3186,0.0095
46,LR,C=32,"(1, 3)",0.2109,0.3136,0.0070
1,NB,"alpha=0.01, variant=complement","(1, 1)",0.2727,0.2903,0.0023
18,NB,"alpha=0.01, variant=complement","(1, 2)",0.2158,0.2856,0.0077
35,NB,"alpha=0.01, variant=complement","(1, 3)",0.2148,0.2779,0.0109
14,SVM,C=0.1,"(1, 1)",0.2924,0.3194,0.0074
32,SVM,C=0.3,"(1, 2)",0.2295,0.3233,0.0057
49,SVM,C=0.3,"(1, 3)",0.2175,0.3207,0.0073


# 5. Técnicas para las clases minoritarias
### Deduplicado y enfoque jerárquico
Cada modelo con su mejor configuración, cruzando las tres opciones de deduplicado con el enfoque plano y el jerárquico (`fallback`).

In [14]:
def run_dedup_hier():
    rows = []
    for model, cfg in BEST_WORD.items():
        feats = vectorize_folds(cfg)
        for dedup, hier in itertools.product(["none", "full", "smart"], [False, True]):
            c = {**cfg, "dedup": dedup, "hier": hier, "root_mode": "fallback"}
            m = evaluate_oof(run_cv(c, feats), c)
            rows.append({"modelo": model, "dedup": dedup, "jerarquico": hier, **m})
            print(f"{model:3s} dedup={dedup:5s} jerárquico={hier!s:5s} F1={m['f1']:.4f}")
    return pd.DataFrame(rows)

dedup_hier = experiment("dedup_jerarquico", run_dedup_hier)
display(dedup_hier.pivot_table(index="modelo", columns=["jerarquico", "dedup"], values="f1")
        .reindex(index=["NB", "LR", "SVM"], columns=pd.MultiIndex.from_product([[False, True], ["none", "full", "smart"]]))
        .round(4))

False                   True                 
          none    full   smart    none    full   smart
modelo                                                
NB      0.2862  0.2848  0.2903  0.2740  0.2732  0.2789
LR      0.3187  0.3194  0.3186  0.3067  0.3076  0.3073
SVM     0.3227  0.3244  0.3233  0.3114  0.3129  0.3123

### Variante `union`
Raíces de las reglas unidas a las del modelo de raíces. En el holdout se mide de dónde salen las raíces de la SVM.

In [15]:
def holdout_fit(cfg, oof):
    keep = copies_to_keep(cfg["dedup"])[dev_idx]
    rep = np.repeat(np.arange(len(dev_idx)), keep)
    vec = make_vectorizer(cfg)
    X_dev, X_ho = vec.fit_transform(TEXTS[dev_idx]), vec.transform(TEXTS[ho_idx])
    res = fit_predict(cfg, X_dev[rep], Y_dev[rep], X_ho)
    thr = tune_thresholds(Y_dev[:, SEC_IDX] if cfg["hier"] else Y_dev, oof["S"])
    return res, thr, decode(res, thr, cfg)

def run_union():
    rows, diag = [], {}
    for model, cfg in BEST_WORD.items():
        c = {**cfg, "hier": True, "root_mode": "union"}
        oof = run_cv(c, vectorize_folds(c))
        rows.append({"modelo": model, "ngram": str(c["ngram"]), "params": params_str(c["params"]), **evaluate_oof(oof, c)})
        if model == "SVM":
            res, thr, P = holdout_fit(c, oof)
            M = res["S"] - thr
            P_sec = force_alias(cap_labels((M >= 0).astype(int), M, MAX_LABELS - 1), SEC)
            root_true = Y_ho[:, ROOT_IDX]
            diag = {"% raíz por reglas": 100 * ((P_sec @ res["R"]) > 0).any(1).mean(),
                    "% con secundaria real": 100 * (Y_ho[:, SEC_IDX].sum(1) > 0).mean(),
                    "% raíces exactas (union)": 100 * (root_true == P[:, ROOT_IDX]).all(1).mean(),
                    "% raíces exactas (solo modelo de raíces)": 100 * (root_true == res["root"]).all(1).mean()}
    df = pd.DataFrame(rows)
    pd.DataFrame([diag]).to_csv(os.path.join(OUT_DIR, "diagnostico_jerarquico.csv"), index=False)
    return df

union = experiment("jerarquico_union", run_union)
display(union[["modelo", "ngram", "params"] + METRICS].round(4))
diag_path = os.path.join(OUT_DIR, "diagnostico_jerarquico.csv")
if os.path.exists(diag_path):
    display(pd.read_csv(diag_path).round(1))

,modelo,ngram,params,f1,f1_std,accuracy,precision,recall,hamming_loss
0,NB,"(1, 1)","alpha=0.01, variant=complement",0.2834,0.0022,0.3929,0.2487,0.3604,0.0391
1,LR,"(1, 2)",C=8,0.3127,0.0082,0.3947,0.2624,0.4604,0.0379
2,SVM,"(1, 2)",C=0.3,0.3179,0.0055,0.4010,0.2667,0.4571,0.0376


# 6. Representaciones por debajo de la palabra
### N-gramas de caracteres
Se repite la búsqueda completa añadiendo un TF-IDF de n-gramas de 2 a 5 caracteres (`char_wb`).

In [16]:
grid_car = experiment("grid_caracteres", lambda: run_grid({**BASE, "char": True}))
BEST_CHAR = best_configs(grid_car, {**BASE, "char": True})

best_w = grid_pal.groupby("modelo")["f1"].max()
best_c = grid_car.groupby("modelo")["f1"].max()
display(pd.DataFrame({"Palabras": best_w, "Palabras + caracteres": best_c, "Diferencia": best_c - best_w})
        .loc[["NB", "LR", "SVM"]].T.round(4))

modelo,NB,LR,SVM
Palabras,0.2903,0.3186,0.3233
Palabras + caracteres,0.2942,0.3270,0.3288
Diferencia,0.0039,0.0084,0.0055


### Subpalabras WordPiece
La mejor configuración de palabras de cada modelo, tokenizando con un vocabulario WordPiece entrenado sobre los textos.

In [17]:
def run_wordpiece():
    rows = []
    for model, cfg in BEST_WORD.items():
        c = {**cfg, "tokenizer": "wordpiece"}
        rows.append({"modelo": model, "ngram": str(c["ngram"]), "params": params_str(c["params"]),
                     **evaluate_oof(run_cv(c, vectorize_folds(c)), c)})
    return pd.DataFrame(rows)

wp = experiment("wordpiece", run_wordpiece)
if wp is not None:
    display(wp.set_index("modelo")[["f1"]].rename(columns={"f1": "WordPiece"})
              .join(best_w.rename("Palabras")).join(best_c.rename("Palabras + caracteres")).round(4))

# 7. Modelo final
Mejor configuración de cada modelo con palabras + caracteres, reentrenada con todo dev y evaluada en el holdout junto al baseline 1 del enunciado.

In [18]:
FINAL_OOF = {}

def run_holdout():
    rows = []
    for model, cfg in BEST_CHAR.items():
        FINAL_OOF[model] = oof = run_cv(cfg, vectorize_folds(cfg))
        _, _, P = holdout_fit(cfg, oof)
        rows.append({"modelo": model, "ngram": str(cfg["ngram"]), "params": params_str(cfg["params"]),
                     **compute_metrics(Y_ho, P)})
    cv5k = CountVectorizer(max_features=5000, stop_words="english")
    b1 = OneVsRestClassifier(MultinomialNB()).fit(cv5k.fit_transform(TEXTS[dev_idx]), Y_dev)
    P_b1 = (b1.predict_proba(cv5k.transform(TEXTS[ho_idx])) >= 0.5).astype(int)
    rows.append({"modelo": "Baseline 1 (MNB)", "ngram": "(1, 1)", "params": "umbral 0.5", **compute_metrics(Y_ho, P_b1)})
    return pd.DataFrame(rows)

ho = experiment("holdout", run_holdout)
display(ho[["modelo", "ngram", "params", "f1", "precision", "recall", "accuracy", "hamming_loss"]].round(4))

,modelo,ngram,params,f1,precision,recall,accuracy,hamming_loss
0,NB,"(1, 1)","alpha=0.01, variant=complement",0.2937,0.2798,0.4302,0.3710,0.0358
1,LR,"(1, 1)",C=2,0.3236,0.2844,0.4361,0.3726,0.0355
2,SVM,"(1, 2)",C=0.05,0.3198,0.2649,0.4975,0.3838,0.0352
3,Baseline 1 (MNB),"(1, 1)",umbral 0.5,0.2200,0.1653,0.6408,0.1778,0.0687


# 8. Envío a Codabench
SVM con palabras (1,2) + caracteres, reentrenada con todos los artículos y con los umbrales ajustados *out-of-fold* en dev.

In [19]:
if RUN["envio"]:
    cfg = BEST_CHAR["SVM"]
    oof = FINAL_OOF.get("SVM") or run_cv(cfg, vectorize_folds(cfg))
    thr = tune_thresholds(Y_dev, oof["S"])

    test = pd.read_csv(TEST_FILE)
    vec = make_vectorizer(cfg)
    X_all, X_te = vec.fit_transform(TEXTS), vec.transform(test["titles"].map(clean) + ". " + test["summaries"].map(clean))
    rep_all = np.repeat(np.arange(N), copies_to_keep(cfg["dedup"]))
    res = fit_predict(cfg, X_all[rep_all], Y[rep_all], X_te)
    P_te = decode_flat(res["S"], thr)

    sub = test[["titles", "summaries"]].copy()
    sub["terms"] = [str(list(t)) for t in mlb.inverse_transform(P_te)]
    sub_dir = os.path.join(OUT_DIR, "envio_svm")
    os.makedirs(sub_dir, exist_ok=True)
    csv_path = os.path.join(sub_dir, "dataset_test_preds.csv")
    sub.to_csv(csv_path, index=False)
    with zipfile.ZipFile(os.path.join(sub_dir, "envio_svm.zip"), "w", zipfile.ZIP_DEFLATED) as z:
        z.write(csv_path, arcname="dataset_test_preds.csv")
    np.save(os.path.join(sub_dir, "test_scores.npy"), res["S"])
    np.save(os.path.join(sub_dir, "thresholds.npy"), thr)
    with open(os.path.join(sub_dir, "config.json"), "w") as fh:
        json.dump({k: (list(v) if isinstance(v, tuple) else v) for k, v in cfg.items()}, fh, indent=2)

    assert len(sub) == len(test) and sub["terms"].map(lambda s: len(ast.literal_eval(s)) > 0).all()
    print(f"Etiquetas por artículo en test: {P_te.sum(1).mean():.2f}")
    display(sub.head())